# Mini-Pilot Studie B Analysis
(2-Stage Verifictaion)

In [ ]:
# Mini-Pilot: Preregistered Evaluation
# CSV schema:
# - dialogA_true/dialogB_true: "good"/"bad"
# - forced_choice: "Dialog A"/"Dialog B"
# - imc_pass: 1/0 (or True/False)
# - pair_no = within-participant position (1st vs 2nd pair)
# - pair_index = global stimulus-pair index (0..16)

import pandas as pd
import numpy as np

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

In [ ]:
# Load data
file_path = "/content/drive/MyDrive/Masterarbeit/Dialoge/StudyB/Mini_Pilot/pilot_all_ratings.csv"
df_raw = pd.read_csv(file_path)

print("Total rows:", len(df_raw))
print("Columns:", df_raw.columns.tolist())
print("Unique pair_no (within-person):", sorted(df_raw["pair_no"].dropna().unique().tolist()))
print("Unique pair_index (global pairs):", sorted(df_raw["pair_index"].dropna().unique().tolist()))
print("N pair_no:", df_raw["pair_no"].nunique(), "| N pair_index:", df_raw["pair_index"].nunique())

Total rows: 68
Columns: ['timestamp', 'rater_id', 'imc_pass', 'pair_no', 'pair_index', 'rank_score', 'recipe_title', 'order', 'dialog_id_good', 'dialog_id_bad', 'dialogA_true', 'dialogB_true', 'dialog_id_A', 'dialog_id_B', 'dqA_overall', 'dqB_overall', 'forced_choice', 'comment']
Unique pair_no (within-person): [1, 2]
Unique pair_index (global pairs): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
N pair_no: 2 | N pair_index: 17


In [ ]:
# Normalise IMC + labels robustly
df = df_raw.copy()

# imc_pass may be 1/0 or True/False
df["imc_pass_bool"] = df["imc_pass"].astype(str).str.strip().str.lower().isin(["1", "true", "t", "yes"])

# dialogA_true/dialogB_true: normalize to "good"/"bad"
df["dialogA_true"] = df["dialogA_true"].astype(str).str.strip().str.lower()
df["dialogB_true"] = df["dialogB_true"].astype(str).str.strip().str.lower()

# forced_choice: normalize to "dialog a"/"dialog b"
df["forced_choice_norm"] = df["forced_choice"].astype(str).str.strip().str.lower()

In [ ]:
# Define exclusion rows (data quality)
excluded_rows = df[
    (~df["imc_pass_bool"]) |
    (df["dqA_overall"].isna()) |
    (df["dqB_overall"].isna()) |
    (df["forced_choice"].isna())
].copy()

included = df.drop(excluded_rows.index).copy()

print("\nIMC failures:", (~df["imc_pass_bool"]).sum())
print("Missing ratings:", (df["dqA_overall"].isna() | df["dqB_overall"].isna()).sum())
print("Missing forced choice:", df["forced_choice"].isna().sum())
print("Total excluded rows:", len(excluded_rows))
print("Total included rows:", len(included))

if len(excluded_rows) > 0:
    display(excluded_rows[[
        "timestamp","rater_id","pair_no","pair_index","recipe_title",
        "imc_pass","dqA_overall","dqB_overall","forced_choice"
    ]].head(50))


IMC failures: 0
Missing ratings: 0
Missing forced choice: 0
Total excluded rows: 0
Total included rows: 68


In [ ]:
# Consistency checks (A/B mapping to good/bad dialog ids)
consA = (
    ((included["dialogA_true"] == "good") &
     (included["dialog_id_A"] == included["dialog_id_good"]) &
     (included["dialog_id_B"] == included["dialog_id_bad"])) |
    ((included["dialogA_true"] == "bad") &
     (included["dialog_id_A"] == included["dialog_id_bad"]) &
     (included["dialog_id_B"] == included["dialog_id_good"]))
)

if not consA.all():
    bad_rows = included.loc[~consA, [
        "rater_id","pair_no","pair_index","dialogA_true",
        "dialog_id_A","dialog_id_B","dialog_id_good","dialog_id_bad"
    ]]
    raise ValueError("Inconsistent A/B dialog-id mapping found:\n" + bad_rows.to_string(index=False))

In [ ]:
# Reconstruct good vs bad ratings and forced_good
def compute_good_bad(row):
    if row["dialogA_true"] == "good":
        good_rating = row["dqA_overall"]
        bad_rating  = row["dqB_overall"]
        forced_good = (row["forced_choice_norm"] == "dialog a")
    elif row["dialogA_true"] == "bad":
        good_rating = row["dqB_overall"]
        bad_rating  = row["dqA_overall"]
        forced_good = (row["forced_choice_norm"] == "dialog b")
    else:
        raise ValueError(f"Unexpected dialogA_true value: {row['dialogA_true']}")
    return pd.Series([good_rating, bad_rating, forced_good])

included[["good_rating","bad_rating","forced_good"]] = included.apply(compute_good_bad, axis=1)
included["diff"] = included["good_rating"] - included["bad_rating"]
included["good_higher"] = included["diff"] > 0

print("\nSANITY (overall):")
print("Mean good_rating:", round(included["good_rating"].mean(), 3))
print("Mean bad_rating :", round(included["bad_rating"].mean(), 3))
print("Overall mean diff:", round(included["diff"].mean(), 3))
print("Overall forced_good proportion:", round(included["forced_good"].mean(), 3))
print("Overall within-person consistency:", round(included["good_higher"].mean(), 3))

# sanity by pair_no (1st vs 2nd within participant)
print("\nSANITY by pair_no (within-person order slot):")
display(included.groupby("pair_no").agg(
    n=("diff","count"),
    mean_diff=("diff","mean"),
    forced_good_prop=("forced_good","mean")
).reset_index().sort_values("pair_no"))


SANITY (overall):
Mean good_rating: 6.25
Mean bad_rating : 1.574
Overall mean diff: 4.676
Overall forced_good proportion: 1.0
Overall within-person consistency: 1.0

SANITY by pair_no (within-person order slot):


,pair_no,n,mean_diff,forced_good_prop
0,1,35,4.571429,1.0
1,2,33,4.787879,1.0


In [ ]:
# Aggregate per global pair (pair_index)
pair_summary = included.groupby("pair_index").agg(
    recipe_title=("recipe_title", "first"),
    rank_score=("rank_score", "first"),
    dialog_id_good=("dialog_id_good", "first"),
    dialog_id_bad=("dialog_id_bad", "first"),
    mean_diff=("diff", "mean"),
    n_evaluations=("diff", "count"),
    forced_good_prop=("forced_good", "mean"),
    within_consistency=("good_higher", "mean")
).reset_index()

In [ ]:
# Apply prereg decision rules (per pair_index)
def evaluate_pair(row):
    if row["n_evaluations"] < 3:
        return "EXCLUDE (insufficient data)"

    direction_correct = (row["mean_diff"] > 0) and (row["forced_good_prop"] > 0.5)

    if (row["mean_diff"] >= 0.5) and (row["forced_good_prop"] >= 0.6):
        return "RETAIN (both thresholds met)"

    if (row["mean_diff"] < 0.5) and (row["forced_good_prop"] < 0.6):
        return "EXCLUDE (both thresholds missed)"

    # Borderline rule
    if direction_correct and (row["within_consistency"] >= 0.6):
        return "RETAIN (borderline rule applied)"

    return "EXCLUDE (borderline failed)"

pair_summary["decision"] = pair_summary.apply(evaluate_pair, axis=1)

# helpful sorting: by pair_index (0..16)
pair_summary = pair_summary.sort_values(by="pair_index")

print("\nMINI-PILOT RESULTS (per pair_index / global stimulus pair)\n")
display(pair_summary)

print("\nSUMMARY")
print(pair_summary["decision"].value_counts())


MINI-PILOT RESULTS (per pair_index / global stimulus pair)



,pair_index,recipe_title,rank_score,dialog_id_good,dialog_id_bad,mean_diff,n_evaluations,forced_good_prop,within_consistency,decision
0,0,Orange Beef Kabobs with Grilled Fruit,3.904343,D021,D063,4.75,4,1.0,1.0,RETAIN (both thresholds met)
1,1,Red Bliss Potato Salad with Gorgonzola and Wal...,1.854148,D007,D048,4.75,4,1.0,1.0,RETAIN (both thresholds met)
2,2,Sweet and Nutty Pork Chops,1.444109,D025,D070,5.25,4,1.0,1.0,RETAIN (both thresholds met)
3,3,Bourbon Wieners,0.886623,D002,D042,5.00,4,1.0,1.0,RETAIN (both thresholds met)
4,4,Turkey Tom Kha Gai,0.476584,D024,D069,4.50,4,1.0,1.0,RETAIN (both thresholds met)
5,5,Mock Caprese Salad,0.476584,D033,D062,4.75,4,1.0,1.0,RETAIN (both thresholds met)
6,6,Apple Pie Filling,0.066545,D040,D077,4.00,4,1.0,1.0,RETAIN (both thresholds met)
7,7,Carrot and Sweet Potato Tzimmes,0.066545,D037,D071,4.00,4,1.0,1.0,RETAIN (both thresholds met)
8,8,Carrot Casserole,0.066545,D019,D060,5.50,4,1.0,1.0,RETAIN (both thresholds met)
9,9,Mom's Margarine Cake,0.066545,D014,D055,4.25,4,1.0,1.0,RETAIN (both thresholds met)



SUMMARY
decision
RETAIN (both thresholds met)    17
Name: count, dtype: int64


In [ ]:
# Show excluded pairs + which dialogs belong to them
excluded_pairs = pair_summary[pair_summary["decision"].str.contains("EXCLUDE")].copy()

print("\nEXCLUDED PAIRS\n")
display(excluded_pairs)

excluded_pair_indices = excluded_pairs["pair_index"].tolist()
print("Excluded pair_index values:", excluded_pair_indices)

if len(excluded_pair_indices) > 0:
    excluded_dialogs = pair_summary[pair_summary["pair_index"].isin(excluded_pair_indices)][
        ["pair_index","recipe_title","dialog_id_good","dialog_id_bad","mean_diff","forced_good_prop","within_consistency","n_evaluations","decision"]
    ].drop_duplicates()
    print("\nEXCLUDED DIALOGS (per excluded pair_index)\n")
    display(excluded_dialogs)
else:
    print("\nNo excluded pairs -> therefore no excluded dialogs from pair-level decision rules.")


EXCLUDED PAIRS



,pair_index,recipe_title,rank_score,dialog_id_good,dialog_id_bad,mean_diff,n_evaluations,forced_good_prop,within_consistency,decision


Excluded pair_index values: []

No excluded pairs -> therefore no excluded dialogs from pair-level decision rules.
